# Specimen 01 — LLM API Basics

Goal: stop training models from scratch, start steering an existing one. This is the first phase where every call has a real, metered cost — build the habit of tracking it now, not in Phase 6 when it's expensive to have skipped.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])


## 1. Set up API access

Create a key at console.anthropic.com, put it in `.env` as `ANTHROPIC_API_KEY=...` (never hardcode it in a cell).

In [ ]:
MODEL = 'claude-opus-5'

assert os.environ.get('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY in .env before continuing'
print('API key loaded, starts with:', os.environ['ANTHROPIC_API_KEY'][:10] + '...')

## 2. First call

Send one prompt (summarize a short article, or answer a factual question). Print the raw response.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "Summarize the following in 2 sentences: The Wright brothers made the first powered, controlled flight in 1903 near Kitty Hawk, North Carolina, using an aircraft they designed and built themselves. Their success came after years of experimenting with gliders and studying aerodynamics, and it launched the era of powered flight."}
    ]
)

for block in response.content:
    if block.type == 'text':
        print(block.text)

print('\nstop_reason:', response.stop_reason)
print('usage:', response.usage)

## 3. Track usage

Pull input/output token counts from the response, compute the actual dollar cost of that call using the model's published pricing.

In [ ]:
INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    input_cost = usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK
    output_cost = usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK
    return input_cost + output_cost

cost = call_cost(response.usage)
print(f'Input tokens: {response.usage.input_tokens}')
print(f'Output tokens: {response.usage.output_tokens}')
print(f'Cost: ${cost:.6f}')

## 4. Build a reusable wrapper

A function that takes a prompt, calls the API, and returns both the text and a running total cost across every call made in this notebook so far.

In [ ]:
session_cost = 0.0
call_log = []

def ask(prompt, max_tokens=300):
    global session_cost
    response = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    text = ''.join(block.text for block in response.content if block.type == 'text')
    cost = call_cost(response.usage)
    session_cost += cost
    call_log.append({'prompt': prompt, 'cost': cost, 'usage': response.usage})
    return text, response

text, response = ask("What year did the Wright brothers first fly?")
print(text)
print(f'\nThis call: ${call_cost(response.usage):.6f}')
print(f'Running session total: ${session_cost:.6f}')

## 5. Multi-turn conversation

Send a follow-up message referencing the first, using the API's message-history format. Confirm the model actually has context from the earlier turn.

In [ ]:
messages = [
    {"role": "user", "content": "My favorite color is teal. Remember that."},
]
response1 = client.messages.create(model=MODEL, max_tokens=100, messages=messages)
reply1 = ''.join(b.text for b in response1.content if b.type == 'text')
print('Turn 1:', reply1)
session_cost += call_cost(response1.usage)

messages.append({"role": "assistant", "content": response1.content})
messages.append({"role": "user", "content": "What's my favorite color?"})

response2 = client.messages.create(model=MODEL, max_tokens=100, messages=messages)
reply2 = ''.join(b.text for b in response2.content if b.type == 'text')
print('Turn 2:', reply2)
session_cost += call_cost(response2.usage)

print('\nContext retained:', 'teal' in reply2.lower())
print(f'Running session total: ${session_cost:.6f}')

## 6. Handle a failure on purpose

Trip a bad parameter or rate limit deliberately, see what the SDK raises, add basic retry/backoff around it.

In [ ]:
import time

def ask_with_retry(prompt, max_tokens=300, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.messages.create(
                model=MODEL,
                max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
            )
        except anthropic.RateLimitError:
            wait = 2 ** attempt
            print(f'Rate limited, retrying in {wait}s (attempt {attempt + 1}/{max_retries})')
            time.sleep(wait)
    raise RuntimeError('Max retries exceeded')

try:
    client.messages.create(
        model=MODEL,
        max_tokens=-1,
        messages=[{"role": "user", "content": "Hello"}],
    )
except anthropic.BadRequestError as e:
    print('Triggered on purpose:', e.status_code, e.message)

print(f'\nFinal session total: ${session_cost:.6f}')